# 🚀 在 Google Colab 上使用 vLLM 部署 Hugging Face 开源模型

> **运行前注意**：请在 Colab 顶部菜单依次点击： 或  ->  -> 选择 **T4 GPU**（免费可用），保存后再运行以下单元格。

## 步骤 1：检查 GPU 显存环境

In [ ]:
# 检查 GPU 环境并清理历史遗留后台进程以释放显存
!pkill -9 -i -f vllm 2>/dev/null || true
!nvidia-smi

## 步骤 2：安装 vLLM 与 Hugging Face 工具库

In [ ]:
!pip install -q -U vllm huggingface_hub openai

## 步骤 3：从 Hugging Face 下载轻量开源模型
这里我们选用 **Qwen/Qwen2.5-0.5B-Instruct**（约 940MB，下载极快，同时具备完整的对话指令微调与现代 Transformer 架构）。

In [ ]:
from huggingface_hub import snapshot_download

model_name = "Qwen/Qwen2.5-0.5B-Instruct"
local_dir = "./Qwen2.5-0.5B-Instruct"

print(f"正在从 Hugging Face 下载模型 {model_name}...")
snapshot_download(repo_id=model_name, local_dir=local_dir)
print()
print("✅ 下载完成！")


## 步骤 4：精简剖析模型（Safetensors 权重、Config 超参、Vocab 词表）

无需加载沉重权重进显存，通过只读 Safetensors Header 与 JSON 配置，1 毫秒内紧凑输出核心参数量、GQA 机制与特殊 Token 映射。


In [ ]:
import os, json, struct

print(f"📁 模型目录: {local_dir}")
print("-" * 65)

# 1. Config 核心架构
with open(os.path.join(local_dir, "config.json")) as f:
    cfg = json.load(f)
q_h, kv_h = cfg.get("num_attention_heads", 0), cfg.get("num_key_value_heads", 0)
gqa_str = f"GQA ({q_h//kv_h}:1)" if kv_h and q_h != kv_h else ("MHA" if q_h == kv_h else "Unknown")

print("【1. Config 核心超参】")
print(f"  • 架构类型 : {cfg.get('architectures', ['Unknown'])[0]} ({cfg.get('model_type', '')})")
print(f"  • 网络深度 : {cfg.get('num_hidden_layers')} layers | Hidden: {cfg.get('hidden_size')} | Intermediate: {cfg.get('intermediate_size')}")
print(f"  • 注意力头 : Q-heads: {q_h} | KV-heads: {kv_h} -> {gqa_str}")
print(f"  • 上下文长 : {cfg.get('max_position_embeddings', 0):,} tokens | RoPE theta: {cfg.get('rope_theta', 'N/A')}")
print(f"  • 权重精度 : {cfg.get('torch_dtype')} | 词表大小: {cfg.get('vocab_size', 0):,} | Tie Embeddings: {cfg.get('tie_word_embeddings', False)}")

print("-" * 65)

# 2. Safetensors 权重 (毫秒级只读 Header，0 显存占用)
sf_path = os.path.join(local_dir, "model.safetensors")
with open(sf_path, "rb") as f:
    h_len = struct.unpack("<Q", f.read(8))[0]
    header = json.loads(f.read(h_len).decode("utf-8"))
tensors = {k: v for k, v in header.items() if k != "__metadata__"}
total_p = sum(eval("*".join(map(str, v["shape"]))) for v in tensors.values())
dtypes = sorted(list(set(v["dtype"] for v in tensors.values())))
fsize_mb = os.path.getsize(sf_path) / (1024 * 1024)

print(f"【2. Safetensors 权重】 ({fsize_mb:.2f} MB | {len(tensors)} tensors | {dtypes[0]})")
print(f"  • 总参数量 : {total_p:,} ({total_p/1e6:.2f}M / {total_p/1e9:.2f}B)")
print("  • 关键权重形状采样:")
for k in ["model.embed_tokens.weight", "model.layers.0.self_attn.q_proj.weight", "model.layers.0.mlp.gate_proj.weight", "model.norm.weight"]:
    if k in tensors:
        print(f"    - {k:<36}: shape={str(tensors[k]['shape']):<16} ({tensors[k]['dtype']})")
if "lm_head.weight" not in tensors and cfg.get("tie_word_embeddings", False):
    print("    - lm_head.weight                      : (与 embed_tokens 共享权重，无独立张量)")

print("-" * 65)

# 3. Vocab 与分词器
tok_cfg_path = os.path.join(local_dir, "tokenizer_config.json")
with open(tok_cfg_path) as f:
    tok_cfg = json.load(f)
added = tok_cfg.get("added_tokens_decoder", {})

print(f"【3. Vocab 词表与特殊标记】 (词表上限: {cfg.get('vocab_size', 0):,} | 扩展特殊标记: {len(added)})")
spec_tokens = []
for tid in sorted(map(int, added.keys())):
    c = added[str(tid)].get("content", "")
    if any(tag in c for tag in ["<|im_start|>", "<|im_end|>", "<|endoftext|>", "bos", "eos"]):
        spec_tokens.append(f"{c} -> {tid}")
print(f"  • 核心对话标记 : {' | '.join(spec_tokens[:4])}")
last_items = [f"{added[str(tid)]['content']}:{tid}" for tid in sorted(map(int, added.keys()))[:3]]
print(f"  • 常用特殊标记 : {last_items}")
print("-" * 65)


## 步骤 5：实战方法 2 —— 就地修改 vLLM 源码注入自定义 Log（In-Place Patching）

**原理剖析**：`pip install vllm` 编译好的 C++/CUDA 算子是动态库，而引擎核心调度流、Block 显存管理均为**原生 Python 源码**（位于 `site-packages/vllm` 中）。

本单元格将直接定位已安装的 `llm_engine.py`，自动备份并向 `LLMEngine.step()` 解码循环中注入自定义调试 Log，随后动态 Reload 生效！


In [ ]:
import os, sys, importlib, inspect
import vllm

# 1. 动态精确定位 vLLM 核心引擎源码文件 (兼容 vLLM V0 与 V1 架构)
try:
    from vllm.v1.engine.llm_engine import LLMEngine
    engine_file = inspect.getfile(LLMEngine)
except Exception:
    try:
        from vllm.engine.llm_engine import LLMEngine
        engine_file = inspect.getfile(LLMEngine)
    except Exception:
        engine_file = os.path.join(os.path.dirname(vllm.__file__), "engine", "llm_engine.py")

print(f"📍 找到 vLLM 核心引擎源码文件: {engine_file}")

# 2. 建立安全备份（首次运行时备份原始纯净文件）
bak_file = engine_file + ".bak"
if not os.path.exists(bak_file):
    with open(engine_file, "r", encoding="utf-8") as f:
        orig = f.read()
    with open(bak_file, "w", encoding="utf-8") as f:
        f.write(orig)
    print(f"📦 已备份原始纯净源码为: {os.path.basename(bak_file)}")
else:
    # 每次基于备份文件修改，确保幂等性与干净
    with open(bak_file, "r", encoding="utf-8") as f:
        orig = f.read()

# 3. 定位到 LLMEngine.step() 函数入口
target_pattern = "def step(self"
pos = orig.find(target_pattern)
if pos == -1:
    raise RuntimeError(f"未在 {engine_file} 中定位到 'def step(self' 方法！")

eol = orig.find("\n", pos)
# 跳过 docstring（如果有）
doc_start = orig.find('"""', eol)
if doc_start != -1 and doc_start - eol < 80:
    doc_end = orig.find('"""', doc_start + 3)
    insert_pos = orig.find("\n", doc_end) + 1
else:
    insert_pos = eol + 1

# 4. 构造我们要注入的自定义 Log 逻辑（跟踪每一步的解码迭代与执行状态）
custom_log_code = '''
        # >>> [USER CUSTOM LOG INJECTION] >>>
        if not hasattr(self, '_custom_step_count'):
            self._custom_step_count = 0
        self._custom_step_count += 1
        
        # 实时打印每一步解码日志
        print(f"🔥 [vLLM源码实时Log | Step #{self._custom_step_count:03d}] 正在执行底层次解码迭代 (LLMEngine.step)")
        # <<< [USER CUSTOM LOG INJECTION] <<<
'''

patched = orig[:insert_pos] + custom_log_code + orig[insert_pos:]

# 5. 覆写到磁盘 site-packages 中
with open(engine_file, "w", encoding="utf-8") as f:
    f.write(patched)

# 6. 同时为入口层 LLM.generate 注入一次性调用 Log (可选加成)
try:
    import vllm.entrypoints.llm as entrypoint_mod
    llm_file = inspect.getfile(entrypoint_mod.LLM)
    bak_llm = llm_file + ".bak"
    if not os.path.exists(bak_llm):
        with open(llm_file, "r", encoding="utf-8") as f:
            llm_orig = f.read()
        with open(bak_llm, "w", encoding="utf-8") as f:
            f.write(llm_orig)
    else:
        with open(bak_llm, "r", encoding="utf-8") as f:
            llm_orig = f.read()
            
    pos_gen = llm_orig.find("def generate(")
    if pos_gen != -1:
        eol_gen = llm_orig.find("-> list[RequestOutput]:", pos_gen)
        if eol_gen != -1:
            doc_s = llm_orig.find('"""', eol_gen)
            if doc_s != -1:
                doc_e = llm_orig.find('"""', doc_s + 3)
                ins_gen = llm_orig.find("\n", doc_e) + 1
                entry_log = '''
        # >>> [USER CUSTOM LOG INJECTION] >>>
        prompt_cnt = len(prompts) if isinstance(prompts, list) else 1
        print(f"🚀 [vLLM源码实时Log | LLM.generate] 接收到批次推理请求 | Prompts 数量: {prompt_cnt}")
        # <<< [USER CUSTOM LOG INJECTION] <<<
'''
                patched_llm = llm_orig[:ins_gen] + entry_log + llm_orig[ins_gen:]
                with open(llm_file, "w", encoding="utf-8") as f:
                    f.write(patched_llm)
                print(f"📍 同时注入入口日志到: {llm_file} (LLM.generate)")
except Exception as e:
    print(f"注意: 入口级日志注入跳过 ({e})")

# 7. 重新加载相关模块，使当前 Python 会话立即应用最新源码修改
for mod_name in list(sys.modules.keys()):
    if "vllm" in mod_name and ("engine" in mod_name or "entrypoints" in mod_name):
        try:
            importlib.reload(sys.modules[mod_name])
        except Exception:
            pass

print("=" * 65)
print("✅ 成功！已就地修改 vLLM 核心源码 (In-Place Edit Applied)！")
print(f"📁 核心引擎文件: {engine_file}")
print("⚡ 注入点: LLMEngine.step() —— 每次 Token 解码迭代均触发")
print("🎯 运行下一步推理时，你将亲眼看到打上 🔥 标记的实时内核调度 Log！")
print("=" * 65)

## 步骤 6：使用 vLLM 进行极速推理（实时观察注入的源码 Log）

现在运行此单元格，注意控制台输出：除了标准推理结果外，还会实时刷出我们在步骤 5 中注入的 `🔥 [vLLM源码实时Log]` 内部调度日志！


In [ ]:
from vllm import LLM, SamplingParams

# 初始化 vLLM 引擎 (T4 GPU 使用 float16)
llm = LLM(
    model=local_dir,
    dtype="float16",
    gpu_memory_utilization=0.7,
    max_model_len=2048
)

# 设置生成采样参数
sampling_params = SamplingParams(
    temperature=0.7,
    top_p=0.8,
    max_tokens=256
)

# Qwen 的标准 Chat Prompt 模板
prompts = [
    "<|im_start|>system\n你是一个风趣幽默的AI助手。<|im_end|>\n<|im_start|>user\n用一句话解释量子力学。<|im_end|>\n<|im_start|>assistant\n",
    "<|im_start|>system\n你是一个资深架构师。<|im_end|>\n<|im_start|>user\n简述 vLLM 的 PagedAttention 是如何解决显存碎片化问题的？<|im_end|>\n<|im_start|>assistant\n"
]

outputs = llm.generate(prompts, sampling_params)

print()
print("====== 推理结果 ======")
print()
for output in outputs:
    prompt = output.prompt
    generated_text = output.outputs[0].text
    print(f"【Prompt】: {prompt.strip()}")
    print(f"【Output】: {generated_text.strip()}")
    print("-" * 50)

## 步骤 7：启动 vLLM 为 OpenAI 兼容的 HTTP API 服务（日志重定向至文件）


In [ ]:
import subprocess, time, requests, gc, torch

# 释放离线引擎显存，为 API Server 腾出空间
try:
    del llm
    gc.collect()
    torch.cuda.empty_cache()
    print('已成功释放离线引擎显存！')
except NameError:
    pass

# 清理之前的旧后台进程
subprocess.run('pkill -9 -i -f vllm 2>/dev/null || true', shell=True)
time.sleep(2)

# 在后台启动 vLLM OpenAI API Server，指定模型名称、T4 float16 与端口
serve_cmd = (
    f'python3 -m vllm.entrypoints.openai.api_server '
    f'--model {local_dir} '
    f'--served-model-name qwen '
    f'--dtype float16 '
    f'--port 8000 '
    f'--gpu-memory-utilization 0.7 '
    f'--max-model-len 2048 '
    f'--uvicorn-log-level info'
)

print('正在后台启动 vLLM API 服务，并将运行日志重定向至 vllm_server.log ...')
log_file = open('vllm_server.log', 'w')
server_process = subprocess.Popen(
    serve_cmd,
    shell=True,
    stdout=log_file,
    stderr=subprocess.STDOUT
)

# 健康检查等待启动
for i in range(40):
    try:
        r = requests.get('http://localhost:8000/health')
        if r.status_code == 200:
            print(f'✅ vLLM API 服务已在第 {i*2} 秒就绪！端口: 8000')
            break
    except Exception:
        pass
    time.sleep(2)


### 步骤 7.5：查看 vLLM API Server 后台运行日志

通过下面这行命令，可以随时查看 API Server 的启动细节、PagedAttention 块分配和请求处理记录：


In [ ]:
# 查看 vLLM 后台服务最新 30 行日志
!tail -n 30 vllm_server.log


## 步骤 8：使用 OpenAI 官方 Python SDK 发起对话请求


In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="dummy-token"
)

# 动态获取已加载的模型 ID
try:
    models = client.models.list().data
    model_id = models[0].id if models else "qwen"
except Exception:
    model_id = "qwen"

print(f"正在请求模型: {model_id} ...")
response = client.chat.completions.create(
    model=model_id,
    messages=[
        {"role": "system", "content": "你是一个专业的程序员助手。"},
        {"role": "user", "content": "用 Python 写一个反转二叉树的函数。"}
    ],
    temperature=0.7,
    max_tokens=256
)

print()
print("====== API 响应结果 ======")
print()
print(response.choices[0].message.content)
